In [20]:
import os
import re
import json
import pandas as pd
from openai import OpenAI
import dotenv
import gspread

dotenv.load_dotenv()

TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")
SPREADSHEET_ID = os.getenv("GOOGLE_SHEETS_SPREADSHEET_ID")
WORKSHEET_NAME = os.getenv("GOOGLE_SHEETS_WORKSHEET_NAME")
SERVICE_ACCOUNT_FILE = os.getenv("GOOGLE_SERVICE_ACCOUNT_FILE", r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\real-estatescraper-17907aa7453a.json")

client = OpenAI(api_key=TYPHOON_API_KEY, base_url="https://api.opentyphoon.ai/v1")

SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญถอดความหมายและสรุปโครงสร้าง “ประกาศขายอสังหาริมทรัพย์ภาษาไทย” สำหรับนำไปสร้างตารางสรุป 4 คอลัมน์หลัก คือ ประเภท, ราคา, ทำเล, ขนาด

งานของคุณมี 2 ส่วนหลัก
1) ประเมินว่าโพสต์นี้เป็น “เจ้าของขายเอง” หรือ “นายหน้า/เอเจนท์”
2) ดึงรายละเอียดโครงสร้างให้ครบถ้วนที่สุดสำหรับฟิลด์ใน extracted โดยเฉพาะ size_text และ location_text ซึ่งจะถูกใช้เป็น row_dict["ขนาด"] และ row_dict["ทำเล"] จึงต้องเก็บรายละเอียดทุกอย่างเกี่ยวกับขนาดและทำเลที่ปรากฏในประกาศ

การประเมินเจ้าของ:
- ประเมิน is_owner ว่า true ถ้าดูมีลักษณะเจ้าของขายเอง, false ถ้าดูเป็นนายหน้า/เอเจนท์/บริษัท
- ใช้บริบทละเอียด เช่น คำปฏิเสธนายหน้า, บุรุษที่หนึ่ง (“ผม/ฉัน/ดิฉัน/เรา”), โทนแคมเปญหลายยูนิต/โปรโมชัน, คำเชิงเอเจนซี (รับฝากขาย/นายหน้า/โบรกเกอร์/realty/estate/property), ชื่อที่ส่อบริษัท, จำนวนเบอร์โทรหลายเบอร์, ข้อความบ่งชี้บริษัทหรือนายหน้า
- ข้อความ “รับนายหน้า/ยินดีนายหน้า/co-broker” อาจพบในโพสต์เจ้าของได้ จึงไม่นับเป็นลบอัตโนมัติ
- confidence ให้เป็นตัวเลข 0.0–1.0 สะท้อนความมั่นใจ

การดึงข้อมูลโครงสร้างใน extracted:
- property_type:
  - ระบุประเภทอสังหาให้ชัดเจน เช่น “บ้านเดี่ยว”, “บ้านแฝด”, “ทาวน์เฮ้าส์”, “คอนโด”, “อาคารพาณิชย์”, “ที่ดิน”, “วิลล่า”, “อพาร์ตเมนต์”, “โฮมออฟฟิศ”, หรือ “อื่นๆ” ที่ใกล้เคียงที่สุด
- price_text และ price_value_thb:
  - อ่านรูปแบบราคาไทยทุกแบบ เช่น “ราคา 17,900,000 บาท”, “ราคาเพียง 1.6 ล้านบาท”, “ขายเพียง 2,200,000 บาท”, “2.5ล.”, “ราคาขาย 2.88ล้านบาท”, “เหลือเพียง 350,000 บาท”
  - เก็บข้อความราคาเต็มดั้งเดิมไว้ใน price_text
  - แปลงเป็นจำนวนเงินหน่วยบาทใน price_value_thb (float) เช่น “1.6 ล้าน” → 1600000.0, “2.5ล.” → 2500000.0
  - ถ้าไม่มีราคาขายให้ใส่ null ทั้ง price_text และ price_value_thb
- location_text (จะใช้เป็นคอลัมน์ “ทำเล”):
  - รวมข้อความที่บ่งชี้ “ที่ตั้ง/ทำเล” ให้ละเอียดที่สุดจากทุกส่วนของประกาศ
  - รวมชื่อโครงการหรือหมู่บ้าน, เลขที่บ้าน, ซอย, ถนน, ตำบล/แขวง, อำเภอ/เขต, จังหวัด, รวมถึงบริบททำเลสำคัญ เช่น “ใกล้มหาวิทยาลัยแม่โจ้”, “ย่านสุเทพ”, “ช้างคลาน เชียงใหม่”
  - ถ้ามีหลายช่วงข้อความที่เป็นที่ตั้ง ให้รวมต่อกันโดยคั่นด้วย “ | ” ภายในสตริงเดียว
  - ห้ามละรายละเอียดทำเลที่ปรากฏอยู่แล้วในประกาศ ถ้าพบให้รวมทั้งหมดใน location_text
- size_text (จะใช้เป็นคอลัมน์ “ขนาด”):
  - ดึง “ทุกข้อความ” ที่กล่าวถึงขนาดหรือพื้นที่ ทั้งเนื้อที่ดินและพื้นที่ใช้สอย เช่น
    - “พื้นที่ชั้น 146 ตร.ม.”
    - “ขนาดที่ดิน 224 ตร.ม. / 56 ตร.วา”
    - “เนื้อที่ดิน 53.2 ตร.วา”, “ขนาดที่ดิน 280 ตร.ม. / 70 ตร.วา”
    - “ขนาด 40.87 ตร.ม.”, “พื้นที่ใช้สอย 230 ตร.ม.”
    - รูปแบบ “1 ไร่ 2 งาน 30 ตร.วา”, หน่วย “ตร.วา/ตร.ม./ไร่/งาน/sq.m/sqm” และข้อความอื่นที่บ่งชี้ขนาด
  - ถ้ามีหลายช่วง ให้รวมทุกช่วงไว้ในสตริงเดียว คั่นด้วย “ | ” เช่น
    - “พื้นที่ชั้น 122 ตร.ม. | ขนาดที่ดิน 224 ตร.ม. / 56 ตร.วา | พื้นที่ใช้สอย 122 ตร.ม.”
  - ห้ามละข้อความขนาดใด ๆ ที่ปรากฏในประกาศ ถ้ามีให้ดึงทั้งหมด
- bedrooms และ bathrooms:
  - แปลงจำนวนห้องนอน/ห้องน้ำเป็นตัวเลข int ถ้าพบ เช่น “3 ห้องนอน 3 ห้องน้ำ”
  - ถ้าไม่พบให้เป็น null
- อื่น ๆ:
  - size_text และ location_text มีความสำคัญมากเพราะจะไปอยู่ใน row_dict["ขนาด"] และ row_dict["ทำเล"] จึงต้องเก็บให้ละเอียดและครบที่สุด
  - โพสต์อาจมาจากเจ้าของหรือเอเจนท์ แต่คุณต้อง extract โครงสร้างให้เหมือนกัน

รูปแบบคำตอบ:
- ตอบเป็น JSON เดียวบรรทัดเดียวเท่านั้น
- ห้ามใช้ code block
- ห้ามมีข้อความอื่นนอกเหนือ JSON
- ต้องเป็นไปตามสคีมา:
{"is_owner": true/false, "confidence": 0.0-1.0, "evidence_phrases": [string...], "risk_flags": [string...], "extracted": {"property_type": "บ้านเดี่ยว/ทาวน์โฮม/คอนโด/อื่นๆ?", "bedrooms": int|null, "bathrooms": int|null, "size_text": string|null, "location_text": string|null, "price_text": string|null, "price_value_thb": float|null}}"""

PROPERTY_PATTERNS = [
    r"บ้านเดี่ยว", r"บ้านแฝด", r"บ้าน(?!พักคนงาน)", r"คอนโด", r"ทาวน์", r"ทาวน์โฮม",
    r"อาคารพาณิชย์", r"ตึกแถว", r"ที่ดิน", r"โกดัง", r"โรงงาน", r"อพาร์ตเมนต์",
    r"แมนชั่น", r"วิลลา", r"คฤหาสน์", r"โฮมออฟฟิศ", r"สำนักงาน", r"ออฟฟิศ",
    r"\bcondo\b", r"\bhouse\b", r"\btownhouse\b", r"\bland\b", r"\bwarehouse\b",
    r"\bapartment\b", r"\boffice\b", r"\bvilla\b", r"\bmansion\b", r"\bpenthouse\b"
]

RENT_OR_IRRELEVANT_PATTERNS = [
    r"ให้เช่า", r"\bเช่า\b", r"ปล่อยเช่า", r"เช่ารายวัน", r"เช่ารายเดือน",
    r"ค่าเช่า", r"ประกันห้อง", r"มัดจำ"
]

prop_regex = [re.compile(p, flags=re.IGNORECASE) for p in PROPERTY_PATTERNS]
rent_regex = [re.compile(p, flags=re.IGNORECASE) for p in RENT_OR_IRRELEVANT_PATTERNS]

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s.replace("\u200b", " ")).strip()
    s = s.replace("ดูน้อยลง", "")
    return s

def is_real_estate(title: str, details: str, desc: str) -> bool:
    t = normalize_text((title or "") + " " + (details or "") + " " + (desc or ""))
    return any(r.search(t) for r in prop_regex)

def is_sale_post(title: str, details: str, desc: str) -> bool:
    t = normalize_text((title or "") + " " + (details or "") + " " + (desc or ""))
    if any(r.search(t) for r in rent_regex):
        return False
    return True

def clamp_text(s: str, n: int = 3500) -> str:
    if not isinstance(s, str):
        return ""
    return s[:n]

def build_user_message(row):
    return (
        "URL: " + clamp_text(row.get("URL", "")) +
        "\nTITLE: " + clamp_text(row.get("Title", "")) +
        "\nPRICE: " + clamp_text(row.get("Price", "")) +
        "\nDETAILS: " + clamp_text(row.get("Property_Details", "")) +
        "\nDESCRIPTION:\n" + clamp_text(row.get("Description", ""))
    )

def extract_json_blob(s: str) -> str:
    s = s.strip().replace("\u200b", "")
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE).strip()
    i = s.find("{")
    j = s.rfind("}")
    if i != -1 and j != -1 and j >= i:
        return s[i:j + 1]
    return s

def parse_float(s: str) -> float:
    m = re.search(r'"?confidence"?\s*:\s*([0-9]+(?:\.[0-9]+)?)', s)
    return float(m.group(1)) if m else 0.0

def parse_extracted(s: str) -> dict:
    d = {}
    jb = extract_json_blob(s)
    m = re.search(r'"?extracted"?\s*:\s*\{(.*)\}', jb, re.S)
    block = m.group(1) if m else ""
    def grab_str(k):
        m2 = re.search(rf'"?{k}"?\s*:\s*"([^"]*)"', block)
        if m2:
            return m2.group(1)
        m3 = re.search(rf"'{k}'\s*:\s*'([^']*)'", block)
        return m3.group(1) if m3 else None
    def grab_int_or_null(k):
        m2 = re.search(rf'"?{k}"?\s*:\s*(\d+|null|None)', block, re.I)
        if not m2:
            return None
        v = m2.group(1)
        if v.lower() in ("null", "none"):
            return None
        return int(v)
    def grab_float_or_null(k):
        m2 = re.search(rf'"?{k}"?\s*:\s*(null|None|[0-9]+(?:\.[0-9]+)?)', block, re.I)
        if not m2:
            return None
        v = m2.group(1)
        if v.lower() in ("null", "none"):
            return None
        return float(v)
    d["property_type"] = grab_str("property_type")
    d["bedrooms"] = grab_int_or_null("bedrooms")
    d["bathrooms"] = grab_int_or_null("bathrooms")
    d["size_text"] = grab_str("size_text")
    d["location_text"] = grab_str("location_text")
    d["price_text"] = grab_str("price_text")
    d["price_value_thb"] = grab_float_or_null("price_value_thb")
    return d

def call_typhoon_owner(json_input_text: str) -> str:
    r = client.chat.completions.create(
        model="typhoon-v2.5-30b-a3b-instruct",
        temperature=0.1,
        max_tokens=2500,
        top_p=0.96,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json_input_text}
        ],
    )
    return r.choices[0].message.content

def format_price_thb(n):
    if n is None:
        return "-"
    if n >= 1_000_000:
        v = round(n / 1_000_000, 2)
        s = f"{v}".rstrip("0").rstrip(".")
        return f"{s} ล้านบาท"
    if n >= 1_000:
        v = round(n / 1_000, 0)
        return f"{int(v)} พันบาท"
    return f"{n} บาท"

def build_row(row, owner_score: int, extracted: dict) -> dict:
    ptxt = extracted.get("price_text")
    pval = extracted.get("price_value_thb")
    if isinstance(pval, str):
        cleaned = pval.replace(",", "").strip()
        if cleaned == "" or cleaned.lower() in ("nan", "null", "none"):
            pval = None
        elif re.fullmatch(r"-?\d+(\.\d+)?", cleaned):
            pval = float(cleaned)
        else:
            pval = None
    price_txt = ptxt if (isinstance(ptxt, str) and ptxt.strip() != "") else (format_price_thb(pval) if pval is not None else "-")
    ptype = extracted.get("property_type") or "-"
    size_txt = extracted.get("size_text") or "-"
    beds = extracted.get("bedrooms")
    baths = extracted.get("bathrooms")
    bb = f"{beds} นอน {baths} น้ำ" if (beds is not None or baths is not None) else "-"
    size_field = (size_txt + " " + bb).strip()
    loc = extracted.get("location_text") or "-"
    url = row.get("URL", "-")
    return {"ประเภท": ptype, "ราคา": price_txt, "ทำเล": loc, "ขนาด": size_field, "Owner Score": f"{owner_score}/100", "Link": url}

def get_ws():
    print("gspread_auth: service_account_from_file")
    print("service_account_file:", SERVICE_ACCOUNT_FILE)
    gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
    print("open_spreadsheet:", SPREADSHEET_ID)
    sh = gc.open_by_key(SPREADSHEET_ID)
    names = [w.title for w in sh.worksheets()]
    ws = sh.worksheet(WORKSHEET_NAME) if WORKSHEET_NAME in names else sh.add_worksheet(title=WORKSHEET_NAME, rows="1000", cols="10")
    hdr = ["ประเภท", "ราคา", "ทำเล", "ขนาด", "Owner Score", "Link"]
    vals = ws.get_all_values()
    if not vals or (len(vals) >= 1 and vals[0][:len(hdr)] != hdr):
        print("set_header:", hdr)
        ws.update("A1:F1", [hdr])
    print("worksheet_ready:", WORKSHEET_NAME)
    return ws

def append_row_ws(ws, rowdict: dict):
    row = [rowdict["ประเภท"], rowdict["ราคา"], rowdict["ทำเล"], rowdict["ขนาด"], rowdict["Owner Score"], rowdict["Link"]]
    ws.append_row(row, value_input_option="USER_ENTERED")
    print("sheet_appended:", rowdict["Link"])

def process_and_save(input_path: str, output_path: str, threshold: int = 80, push_to_sheet: bool = True):
    print("load_csv:", input_path)
    df = pd.read_csv(input_path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    if "Post_URL" in df.columns:
        df["URL"] = df["Post_URL"]
    if "Full_Post_Content" in df.columns:
        df["Description"] = df["Full_Post_Content"]
    for c in ["Title", "Property_Details", "Price"]:
        if c not in df.columns:
            df[c] = ""
    ws = get_ws() if push_to_sheet else None
    rows = []
    print("start_iterate")
    for i, row in df.iterrows():
        title = normalize_text(row.get("Title", ""))
        details = normalize_text(row.get("Property_Details", ""))
        desc = normalize_text(row.get("Description", ""))
        if not is_real_estate(title, details, desc):
            if i % 50 == 0:
                print("skip_non_real_estate_index:", i)
            continue
        if not is_sale_post(title, details, desc):
            if i % 50 == 0:
                print("skip_non_sale_index:", i)
            continue
        user_msg = build_user_message({"URL": row.get("URL", ""), "Title": title, "Price": row.get("Price", ""), "Property_Details": details, "Description": desc})
        print("typhoon_call_index:", i)
        raw = call_typhoon_owner(user_msg)
        blob = extract_json_blob(raw)
        ty_conf = parse_float(blob)
        score = int(round(ty_conf * 100))
        if score >= threshold:
            extracted = parse_extracted(blob)
            out = build_row({"URL": row.get("URL", ""), "Title": title, "Price": row.get("Price", "")}, score, extracted)
            rows.append(out)
            print("accepted_index:", i, "score:", score, "price:", out["ราคา"])
            if push_to_sheet:
                append_row_ws(ws, out)
        else:
            if i % 20 == 0:
                print("below_threshold_index:", i, "score:", score)
    print("kept_rows:", len(rows))
    out_df = pd.DataFrame(rows, columns=["ประเภท", "ราคา", "ทำเล", "ขนาด", "Owner Score", "Link"])
    out_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("saved_csv:", output_path)

if __name__ == "__main__":
    input_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_scraped_details.csv"
    output_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\CSV_file\owner_scoredddproperty.csv"
    process_and_save(input_path, output_path, threshold=1, push_to_sheet=True)

load_csv: C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_scraped_details.csv
shape: (784, 2)
columns: ['Post_URL', 'Full_Post_Content']
gspread_auth: service_account_from_file
service_account_file: C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\real-estatescraper-17907aa7453a.json
open_spreadsheet: 19vP9nCxkpAj2Rjqlaiap-pmgGhHHZz-RcWvpxTVk8Ng
set_header: ['ประเภท', 'ราคา', 'ทำเล', 'ขนาด', 'Owner Score', 'Link']


C:\Users\kongl\AppData\Local\Temp\ipykernel_16536\1096876417.py:222: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update("A1:F1", [hdr])


worksheet_ready: kaidee
start_iterate
skip_non_sale_index: 0
typhoon_call_index: 1
accepted_index: 1 score: 95 price: ราคา: 850,000
sheet_appended: https://baan.kaidee.com/product-367612538
typhoon_call_index: 2
accepted_index: 2 score: 95 price: ราคา: 2,990,000
sheet_appended: https://baan.kaidee.com/product-367803193
typhoon_call_index: 3
accepted_index: 3 score: 95 price: 450,000,000
sheet_appended: https://baan.kaidee.com/product-367882394
typhoon_call_index: 4
accepted_index: 4 score: 95 price: 2,790,000
sheet_appended: https://baan.kaidee.com/product-367937725
typhoon_call_index: 5
accepted_index: 5 score: 95 price: ราคาขาย 1.6 ล้านบาท
sheet_appended: https://baan.kaidee.com/product-368123336
typhoon_call_index: 6
accepted_index: 6 score: 95 price: ราคาขาย 1.6 ล้านบาท
sheet_appended: https://baan.kaidee.com/product-368553060
typhoon_call_index: 7
accepted_index: 7 score: 95 price: ราคา: 2,490,000
sheet_appended: https://baan.kaidee.com/product-368556614
typhoon_call_index: 11
acc

In [19]:
import os
import re
import json
import time
import pandas as pd
import google.generativeai as genai
import dotenv
import gspread
from google.oauth2.service_account import Credentials

print("[Stage: Init] Load environment variables")
dotenv.load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
SPREADSHEET_ID = os.getenv("GOOGLE_SHEETS_SPREADSHEET_ID")
WORKSHEET_NAME = os.getenv("GOOGLE_SHEETS_WORKSHEET_NAME")
SERVICE_ACCOUNT_FILE = os.getenv(
    "GOOGLE_SERVICE_ACCOUNT_FILE",
    r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\real-estatescraper-17907aa7453a.json"
)
RESUME_INDEX = 0

print("[Stage: Init] Configure Gemini client")
genai.configure(api_key=GEMINI_API_KEY)

SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญถอดความหมายและสรุปโครงสร้าง “ประกาศขายอสังหาริมทรัพย์ภาษาไทย” สำหรับนำไปสร้างตารางสรุป 4 คอลัมน์หลัก คือ ประเภท, ราคา, ทำเล, ขนาด

งานของคุณมี 2 ส่วนหลัก
1) ประเมินว่าโพสต์นี้เป็น “เจ้าของขายเอง” หรือ “นายหน้า/เอเจนท์”
2) ดึงรายละเอียดโครงสร้างให้ครบถ้วนที่สุดสำหรับฟิลด์ใน extracted โดยเฉพาะ size_text และ location_text ซึ่งจะถูกใช้เป็น row_dict["ขนาด"] และ row_dict["ทำเล"] จึงต้องเก็บรายละเอียดทุกอย่างเกี่ยวกับขนาดและทำเลที่ปรากฏในประกาศ

การประเมินเจ้าของ:
- ประเมิน is_owner ว่า true ถ้าดูมีลักษณะเจ้าของขายเอง, false ถ้าดูเป็นนายหน้า/เอเจนท์/บริษัท
- ใช้บริบทละเอียด เช่น คำปฏิเสธนายหน้า, บุรุษที่หนึ่ง (“ผม/ฉัน/ดิฉัน/เรา”), โทนแคมเปญหลายยูนิต/โปรโมชัน, คำเชิงเอเจนซี (รับฝากขาย/นายหน้า/โบรกเกอร์/realty/estate/property), ชื่อที่ส่อบริษัท, จำนวนเบอร์โทรหลายเบอร์, ข้อความบ่งชี้บริษัทหรือนายหน้า
- ข้อความ “รับนายหน้า/ยินดีนายหน้า/co-broker” อาจพบในโพสต์เจ้าของได้ จึงไม่นับเป็นลบอัตโนมัติ
- confidence ให้เป็นตัวเลข 0.0–1.0 สะท้อนความมั่นใจ

การดึงข้อมูลโครงสร้างใน extracted:
- property_type:
  - ระบุประเภทอสังหาให้ชัดเจน เช่น “บ้านเดี่ยว”, “บ้านแฝด”, “ทาวน์เฮ้าส์”, “คอนโด”, “อาคารพาณิชย์”, “ที่ดิน”, “วิลล่า”, “อพาร์ตเมนต์”, “โฮมออฟฟิศ”, หรือ “อื่นๆ” ที่ใกล้เคียงที่สุด
- price_text และ price_value_thb:
  - อ่านรูปแบบราคาไทยทุกแบบ เช่น “ราคา 17,900,000 บาท”, “ราคาเพียง 1.6 ล้านบาท”, “ขายเพียง 2,200,000 บาท”, “2.5ล.”, “ราคาขาย 2.88ล้านบาท”, “เหลือเพียง 350,000 บาท”
  - เก็บข้อความราคาเต็มดั้งเดิมไว้ใน price_text
  - แปลงเป็นจำนวนเงินหน่วยบาทใน price_value_thb (float) เช่น “1.6 ล้าน” → 1600000.0, “2.5ล.” → 2500000.0
  - ถ้าไม่มีราคาขายให้ใส่ null ทั้ง price_text และ price_value_thb
- location_text (จะใช้เป็นคอลัมน์ “ทำเล”):
  - รวมข้อความที่บ่งชี้ “ที่ตั้ง/ทำเล” ให้ละเอียดที่สุดจากทุกส่วนของประกาศ
  - รวมชื่อโครงการหรือหมู่บ้าน, เลขที่บ้าน, ซอย, ถนน, ตำบล/แขวง, อำเภอ/เขต, จังหวัด, รวมถึงบริบททำเลสำคัญ เช่น “ใกล้มหาวิทยาลัยแม่โจ้”, “ย่านสุเทพ”, “ช้างคลาน เชียงใหม่”
  - ถ้ามีหลายช่วงข้อความที่เป็นที่ตั้ง ให้รวมต่อกันโดยคั่นด้วย “ | ” ภายในสตริงเดียว
  - ห้ามละรายละเอียดทำเลที่ปรากฏอยู่แล้วในประกาศ ถ้าพบให้รวมทั้งหมดใน location_text
- size_text (จะใช้เป็นคอลัมน์ “ขนาด”):
  - ดึง “ทุกข้อความ” ที่กล่าวถึงขนาดหรือพื้นที่ ทั้งเนื้อที่ดินและพื้นที่ใช้สอย เช่น
    - “พื้นที่ชั้น 146 ตร.ม.”
    - “ขนาดที่ดิน 224 ตร.ม. / 56 ตร.วา”
    - “เนื้อที่ดิน 53.2 ตร.วา”, “ขนาดที่ดิน 280 ตร.ม. / 70 ตร.วา”
    - “ขนาด 40.87 ตร.ม.”, “พื้นที่ใช้สอย 230 ตร.ม.”
    - รูปแบบ “1 ไร่ 2 งาน 30 ตร.วา”, หน่วย “ตร.วา/ตร.ม./ไร่/งาน/sq.m/sqm” และข้อความอื่นที่บ่งชี้ขนาด
  - ถ้ามีหลายช่วง ให้รวมทุกช่วงไว้ในสตริงเดียว คั่นด้วย “ | ” เช่น
    - “พื้นที่ชั้น 122 ตร.ม. | ขนาดที่ดิน 224 ตร.ม. / 56 ตร.วา | พื้นที่ใช้สอย 122 ตร.ม.”
  - ห้ามละข้อความขนาดใด ๆ ที่ปรากฏในประกาศ ถ้ามีให้ดึงทั้งหมด
- bedrooms และ bathrooms:
  - แปลงจำนวนห้องนอน/ห้องน้ำเป็นตัวเลข int ถ้าพบ เช่น “3 ห้องนอน 3 ห้องน้ำ”
  - ถ้าไม่พบให้เป็น null
- อื่น ๆ:
  - size_text และ location_text มีความสำคัญมากเพราะจะไปอยู่ใน row_dict["ขนาด"] และ row_dict["ทำเล"] จึงต้องเก็บให้ละเอียดและครบที่สุด
  - โพสต์อาจมาจากเจ้าของหรือเอเจนท์ แต่คุณต้อง extract โครงสร้างให้เหมือนกัน

รูปแบบคำตอบ:
- ตอบเป็น JSON เดียวบรรทัดเดียวเท่านั้น
- ห้ามใช้ code block
- ห้ามมีข้อความอื่นนอกเหนือ JSON
- ต้องเป็นไปตามสคีมา:
{"is_owner": true/false, "confidence": 0.0-1.0, "evidence_phrases": [string...], "risk_flags": [string...], "extracted": {"property_type": "บ้านเดี่ยว/ทาวน์โฮม/คอนโด/อื่นๆ?", "bedrooms": int|null, "bathrooms": int|null, "size_text": string|null, "location_text": string|null, "price_text": string|null, "price_value_thb": float|null}}"""

PROPERTY_PATTERNS = [
    r"บ้านเดี่ยว", r"บ้านแฝด", r"บ้าน(?!พักคนงาน)", r"คอนโด", r"ทาวน์", r"ทาวน์โฮม",
    r"อาคารพาณิชย์", r"ตึกแถว", r"ที่ดิน", r"โกดัง", r"โรงงาน", r"อพาร์ตเมนต์",
    r"แมนชั่น", r"วิลลา", r"คฤหาสน์", r"โฮมออฟฟิศ", r"สำนักงาน", r"ออฟฟิศ",
    r"\bcondo\b", r"\bhouse\b", r"\btownhouse\b", r"\bland\b", r"\bwarehouse\b",
    r"\bapartment\b", r"\boffice\b", r"\bvilla\b", r"\bmansion\b", r"\bpenthouse\b"
]

RENT_OR_IRRELEVANT_PATTERNS = [
    r"ให้เช่า", r"\bเช่า\b", r"ปล่อยเช่า", r"เช่ารายวัน", r"เช่ารายเดือน",
    r"ค่าเช่า", r"ประกันห้อง", r"มัดจำ"
]

def normalize_text(s: str) -> str:
    if not isinstance(s, str): return ""
    return re.sub(r"\s+", " ", s.replace("\u200b", " ").replace("ดูน้อยลง", "")).strip()

def is_relevant_for_sale_post(full_text: str) -> bool:
    if not any(re.search(p, full_text) for p in PROPERTY_PATTERNS): return False
    if any(re.search(p, full_text) for p in RENT_OR_IRRELEVANT_PATTERNS): return False
    return True

def clamp_text(s: str, n: int = 8000) -> str:
    if not isinstance(s, str): return ""
    return s[:n]

def build_user_message(row):
    full_content = " ".join(filter(None, [
        row.get("Title", ""),
        row.get("Price", ""),
        row.get("Property_Details", ""),
        row.get("Description", "")
    ]))
    return clamp_text(normalize_text(full_content))

def call_gemini_analyzer(user_message: str) -> dict:
    model = genai.GenerativeModel(
        'gemini-2.5-flash-lite',
        system_instruction=SYSTEM_PROMPT
    )
    generation_config = genai.types.GenerationConfig(
        response_mime_type="application/json",
        temperature=0.02
    )
    response = model.generate_content(
        user_message,
        generation_config=generation_config
    )
    cleaned_text = re.sub(r"^```json|^```|```$", "", response.text, flags=re.MULTILINE).strip()
    return json.loads(cleaned_text)

def build_row(row_data: dict, llm_result: dict) -> dict:
    extracted = llm_result.get("extracted", {})
    is_owner = llm_result.get("is_owner", False)
    confidence = llm_result.get("confidence", 0.0) if llm_result.get("confidence") is not None else 0.0
    owner_score = int(confidence * 100) if is_owner else int((1 - confidence) * 50)
    price_val = extracted.get("price_value_thb")
    price_text = extracted.get("price_text") or (f"{price_val/1_000_000:.2f} ล้านบาท".rstrip('0').rstrip('.') if price_val and price_val >= 1_000_000 else f"{price_val:,.0f} บาท" if price_val else "-")
    size_field = extracted.get("size_text") or "-"
    location_field = extracted.get("location_text") or "-"
    property_type = extracted.get("property_type") or "-"
    return {
        "ประเภท": property_type,
        "ราคา": price_text,
        "ทำเล": location_field,
        "ขนาด": size_field,
        "Owner Score": f"{owner_score}/100",
        "Link": row_data.get("URL", "-")
    }

def get_worksheet():
    print("[Stage: Sheets] Authorize and open worksheet")
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SPREADSHEET_ID)
    ws = sh.worksheet(WORKSHEET_NAME)
    header = ["ประเภท","ราคา","ทำเล","ขนาด","Owner Score","Link"]
    current_header = ws.get("A1:F1")
    if not current_header or current_header[0] != header:
        ws.update("A1:F1", [header])
        ws.format("A1:F1", {'textFormat': {'bold': True}})
    print("[Stage: Sheets] Worksheet ready")
    return ws

def append_to_worksheet(ws, row_dict: dict):
    row_values = [row_dict["ประเภท"], row_dict["ราคา"], row_dict["ทำเล"], row_dict["ขนาด"], row_dict["Owner Score"], row_dict["Link"]]
    ws.append_row(row_values, value_input_option="USER_ENTERED")
    print(f"sheet_appended: {row_dict['Link']}")
    time.sleep(1.0)

def process_and_save(input_path: str, output_path: str, threshold: int = 80, push_to_sheet: bool = True):
    print("[Stage: Input] Load CSV:", input_path)
    df = pd.read_csv(input_path, encoding="utf-8")
    if "Post_URL" in df.columns: df.rename(columns={"Post_URL": "URL"}, inplace=True)
    if "Full_Post_Content" in df.columns: df.rename(columns={"Full_Post_Content": "Description"}, inplace=True)
    for col in ["Title", "Property_Details", "Price", "URL", "Description"]:
        if col not in df.columns: df[col] = ""
        df[col] = df[col].fillna('')
    ws = get_worksheet() if push_to_sheet else None
    final_rows = []
    print("[Stage: Iterate] Start from index:", RESUME_INDEX)
    for i, row in df.iterrows():
        if i < RESUME_INDEX:
            if i % 200 == 0: print(f"resume_skip_index: {i}")
            continue
        full_text = normalize_text(f"{row.get('Title', '')} {row.get('Property_Details', '')} {row.get('Description', '')}")
        if not is_relevant_for_sale_post(full_text):
            if i % 50 == 0: print(f"skip_non_relevant_index: {i}")
            continue
        user_message = build_user_message(row)
        print(f"gemini_call_index: {i}, URL: {row.get('URL', 'N/A')}")
        llm_result = call_gemini_analyzer(user_message)
        if not llm_result:
            print(f"  - Skip: LLM returned empty result for index {i}")
            continue
        is_owner = llm_result.get("is_owner")
        confidence = llm_result.get("confidence", 0.0) if llm_result.get("confidence") is not None else 0.0
        score = int(confidence * 100) if is_owner else int((1 - confidence) * 50)
        if score >= threshold:
            output_row = build_row(row, llm_result)
            final_rows.append(output_row)
            print(f"  - Accepted: index={i}, score={score}, price={output_row['ราคา']}")
            if push_to_sheet and ws:
                append_to_worksheet(ws, output_row)
        else:
            if i % 20 == 0: print(f"  - Skip: Score below threshold for index {i}, score={score}")
    print("kept_rows:", len(final_rows))
    out_df = pd.DataFrame(final_rows)
    out_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("saved_csv:", output_path)

if __name__ == "__main__":
    input_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_scraped_details.csv"
    output_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\CSV_file\owner_score_gemini_filtered.csv"
    print(f"[Stage: Run] Start pipeline; RESUME_INDEX = {RESUME_INDEX}")
    process_and_save(input_path, output_path, threshold=1, push_to_sheet=True)
    print("[Stage: Done] Completed")

[Stage: Init] Load environment variables
[Stage: Init] Configure Gemini client
[Stage: Run] Start pipeline; RESUME_INDEX = 0
[Stage: Input] Load CSV: C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_scraped_details.csv
[Stage: Sheets] Authorize and open worksheet


C:\Users\kongl\AppData\Local\Temp\ipykernel_16536\854907865.py:158: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update("A1:F1", [header])


[Stage: Sheets] Worksheet ready
[Stage: Iterate] Start from index: 0
skip_non_relevant_index: 0
gemini_call_index: 1, URL: https://baan.kaidee.com/product-367612538


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 

In [ ]:
import os
import re
import json
import time
import pandas as pd
import google.generativeai as genai
import dotenv
import gspread
from google.oauth2.service_account import Credentials

print("[Stage: Init] Load environment variables")
dotenv.load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
SPREADSHEET_ID = os.getenv("GOOGLE_SHEETS_SPREADSHEET_ID")
WORKSHEET_NAME = os.getenv("GOOGLE_SHEETS_WORKSHEET_NAME")
SERVICE_ACCOUNT_FILE = os.getenv("GOOGLE_SERVICE_ACCOUNT_FILE", r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\real-estatescraper-17907aa7453a.json")
RESUME_INDEX = int(os.getenv("RESUME_INDEX", "1514"))

print("[Stage: Init] Configure Gemini client")
genai.configure(api_key=GEMINI_API_KEY)

SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญถอดความหมายและสรุปโครงสร้าง “ประกาศขายอสังหาริมทรัพย์ภาษาไทย” สำหรับนำไปสร้างตารางสรุป 4 คอลัมน์หลัก คือ ประเภท, ราคา, ทำเล, ขนาด

งานของคุณมี 2 ส่วนหลัก
1) ประเมินว่าโพสต์นี้เป็น “เจ้าของขายเอง” หรือ “นายหน้า/เอเจนท์”
2) ดึงรายละเอียดโครงสร้างให้ครบถ้วนที่สุดสำหรับฟิลด์ใน extracted โดยเฉพาะ size_text และ location_text ซึ่งจะถูกใช้เป็น row_dict["ขนาด"] และ row_dict["ทำเล"] จึงต้องเก็บรายละเอียดทุกอย่างเกี่ยวกับขนาดและทำเลที่ปรากฏในประกาศ

การประเมินเจ้าของ:
- ประเมิน is_owner ว่า true ถ้าดูมีลักษณะเจ้าของขายเอง, false ถ้าดูเป็นนายหน้า/เอเจนท์/บริษัท
- ใช้บริบทละเอียด เช่น คำปฏิเสธนายหน้า, บุรุษที่หนึ่ง (“ผม/ฉัน/ดิฉัน/เรา”), โทนแคมเปญหลายยูนิต/โปรโมชัน, คำเชิงเอเจนซี (รับฝากขาย/นายหน้า/โบรกเกอร์/realty/estate/property), ชื่อที่ส่อบริษัท, จำนวนเบอร์โทรหลายเบอร์, ข้อความบ่งชี้บริษัทหรือนายหน้า
- ข้อความ “รับนายหน้า/ยินดีนายหน้า/co-broker” อาจพบในโพสต์เจ้าของได้ จึงไม่นับเป็นลบอัตโนมัติ
- confidence ให้เป็นตัวเลข 0.0–1.0 สะท้อนความมั่นใจ

การดึงข้อมูลโครงสร้างใน extracted:
- property_type:
  - ระบุประเภทอสังหาให้ชัดเจน เช่น “บ้านเดี่ยว”, “บ้านแฝด”, “ทาวน์เฮ้าส์”, “คอนโด”, “อาคารพาณิชย์”, “ที่ดิน”, “วิลล่า”, “อพาร์ตเมนต์”, “โฮมออฟฟิศ”, หรือ “อื่นๆ” ที่ใกล้เคียงที่สุด
- price_text และ price_value_thb:
  - อ่านรูปแบบราคาไทยทุกแบบ เช่น “ราคา 17,900,000 บาท”, “ราคาเพียง 1.6 ล้านบาท”, “ขายเพียง 2,200,000 บาท”, “2.5ล.”, “ราคาขาย 2.88ล้านบาท”, “เหลือเพียง 350,000 บาท”
  - เก็บข้อความราคาเต็มดั้งเดิมไว้ใน price_text
  - แปลงเป็นจำนวนเงินหน่วยบาทใน price_value_thb (float) เช่น “1.6 ล้าน” → 1600000.0, “2.5ล.” → 2500000.0
  - ถ้าไม่มีราคาขายให้ใส่ null ทั้ง price_text และ price_value_thb
- location_text (จะใช้เป็นคอลัมน์ “ทำเล”):
  - รวมข้อความที่บ่งชี้ “ที่ตั้ง/ทำเล” ให้ละเอียดที่สุดจากทุกส่วนของประกาศ
  - รวมชื่อโครงการหรือหมู่บ้าน, เลขที่บ้าน, ซอย, ถนน, ตำบล/แขวง, อำเภอ/เขต, จังหวัด, รวมถึงบริบททำเลสำคัญ เช่น “ใกล้มหาวิทยาลัยแม่โจ้”, “ย่านสุเทพ”, “ช้างคลาน เชียงใหม่”
  - ถ้ามีหลายช่วงข้อความที่เป็นที่ตั้ง ให้รวมต่อกันโดยคั่นด้วย “ | ” ภายในสตริงเดียว
  - ห้ามละรายละเอียดทำเลที่ปรากฏอยู่แล้วในประกาศ ถ้าพบให้รวมทั้งหมดใน location_text
- size_text (จะใช้เป็นคอลัมน์ “ขนาด”):
  - ดึง “ทุกข้อความ” ที่กล่าวถึงขนาดหรือพื้นที่ ทั้งเนื้อที่ดินและพื้นที่ใช้สอย เช่น
    - “พื้นที่ชั้น 146 ตร.ม.”
    - “ขนาดที่ดิน 224 ตร.ม. / 56 ตร.วา”
    - “เนื้อที่ดิน 53.2 ตร.วา”, “ขนาดที่ดิน 280 ตร.ม. / 70 ตร.วา”
    - “ขนาด 40.87 ตร.ม.”, “พื้นที่ใช้สอย 230 ตร.ม.”
    - รูปแบบ “1 ไร่ 2 งาน 30 ตร.วา”, หน่วย “ตร.วา/ตร.ม./ไร่/งาน/sq.m/sqm” และข้อความอื่นที่บ่งชี้ขนาด
  - ถ้ามีหลายช่วง ให้รวมทุกช่วงไว้ในสตริงเดียว คั่นด้วย “ | ” เช่น
    - “พื้นที่ชั้น 122 ตร.ม. | ขนาดที่ดิน 224 ตร.ม. / 56 ตร.วา | พื้นที่ใช้สอย 122 ตร.ม.”
  - ห้ามละข้อความขนาดใด ๆ ที่ปรากฏในประกาศ ถ้ามีให้ดึงทั้งหมด
- bedrooms และ bathrooms:
  - แปลงจำนวนห้องนอน/ห้องน้ำเป็นตัวเลข int ถ้าพบ เช่น “3 ห้องนอน 3 ห้องน้ำ”
  - ถ้าไม่พบให้เป็น null
- อื่น ๆ:
  - size_text และ location_text มีความสำคัญมากเพราะจะไปอยู่ใน row_dict["ขนาด"] และ row_dict["ทำเล"] จึงต้องเก็บให้ละเอียดและครบที่สุด
  - โพสต์อาจมาจากเจ้าของหรือเอเจนท์ แต่คุณต้อง extract โครงสร้างให้เหมือนกัน

รูปแบบคำตอบ:
- ตอบเป็น JSON เดียวบรรทัดเดียวเท่านั้น
- ห้ามใช้ code block
- ห้ามมีข้อความอื่นนอกเหนือ JSON
- ต้องเป็นไปตามสคีมา:
{"is_owner": true/false, "confidence": 0.0-1.0, "evidence_phrases": [string...], "risk_flags": [string...], "extracted": {"property_type": "บ้านเดี่ยว/ทาวน์โฮม/คอนโด/อื่นๆ?", "bedrooms": int|null, "bathrooms": int|null, "size_text": string|null, "location_text": string|null, "price_text": string|null, "price_value_thb": float|null}}"""

PROPERTY_PATTERNS = [
    r"บ้านเดี่ยว", r"บ้านแฝด", r"บ้าน(?!พักคนงาน)", r"คอนโด", r"ทาวน์", r"ทาวน์โฮม",
    r"อาคารพาณิชย์", r"ตึกแถว", r"ที่ดิน", r"โกดัง", r"โรงงาน", r"อพาร์ตเมนต์",
    r"แมนชั่น", r"วิลลา", r"คฤหาสน์", r"โฮมออฟฟิศ", r"สำนักงาน", r"ออฟฟิศ",
    r"\bcondo\b", r"\bhouse\b", r"\btownhouse\b", r"\bland\b", r"\bwarehouse\b",
    r"\bapartment\b", r"\boffice\b", r"\bvilla\b", r"\bmansion\b", r"\bpenthouse\b"
]

RENT_OR_IRRELEVANT_PATTERNS = [
    r"ให้เช่า", r"\bเช่า\b", r"ปล่อยเช่า", r"เช่ารายวัน", r"เช่ารายเดือน",
    r"ค่าเช่า", r"ประกันห้อง", r"มัดจำ"
]

def normalize_text(s: str) -> str:
    if not isinstance(s, str): return ""
    return re.sub(r"\s+", " ", s.replace("\u200b", " ").replace("ดูน้อยลง", "")).strip()

def is_relevant_for_sale_post(full_text: str) -> bool:
    if not any(re.search(r, full_text) for r in PROPERTY_PATTERNS): return False
    if any(re.search(r, full_text) for r in RENT_OR_IRRELEVANT_PATTERNS): return False
    return True

def clamp_text(s: str, n: int = 8000) -> str:
    if not isinstance(s, str): return ""
    return s[:n]

def build_user_message(row):
    full_content = " ".join(filter(None, [
        row.get("Title", ""),
        row.get("Price", ""),
        row.get("Property_Details", ""),
        row.get("Description", "")
    ]))
    return clamp_text(normalize_text(full_content))

def call_gemini_analyzer(user_message: str) -> dict:
    model = genai.GenerativeModel(
        'gemini-2.5-flash-lite',
        system_instruction=SYSTEM_PROMPT
    )
    generation_config = genai.types.GenerationConfig(
        response_mime_type="application/json",
        temperature=0.05
    )
    safety_settings = {
        'HARM_CATEGORY_HARASSMENT': 'BLOCK_NONE',
        'HARM_CATEGORY_HATE_SPEECH': 'BLOCK_NONE',
        'HARM_CATEGORY_SEXUALLY_EXPLICIT': 'BLOCK_NONE',
        'HARM_CATEGORY_DANGEROUS_CONTENT': 'BLOCK_NONE',
    }
    
    response = model.generate_content(
        user_message,
        generation_config=generation_config,
        safety_settings=safety_settings
    )
    
    return json.loads(response.text)

def build_row(row_data: dict, llm_result: dict) -> dict:
    extracted = llm_result.get("extracted", {})
    is_owner = llm_result.get("is_owner", False)
    confidence = llm_result.get("confidence", 0.0) if llm_result.get("confidence") is not None else 0.0
    
    owner_score = int(confidence * 100) if is_owner else int((1 - confidence) * 50)
    
    price_val = extracted.get("price_value_thb")
    price_text = extracted.get("price_text") or (f"{price_val/1_000_000:.2f} ล้านบาท".rstrip('0').rstrip('.') if price_val and price_val >= 1_000_000 else f"{price_val:,.0f} บาท" if price_val else "-")

    size_field = extracted.get("size_text") or "-"
    location_field = extracted.get("location_text") or "-"
    property_type = extracted.get("property_type") or "-"
    
    return {
        "ประเภท": property_type,
        "ราคา": price_text,
        "ทำเล": location_field,
        "ขนาด": size_field,
        "Owner Score": f"{owner_score}/100",
        "Link": row_data.get("URL", "-")
    }

def get_worksheet():
    print("[Stage: Sheets] Authorize and open worksheet")
    scopes = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SPREADSHEET_ID)
    ws = sh.worksheet(WORKSHEET_NAME)
    
    header = ["ประเภท","ราคา","ทำเล","ขนาด","Owner Score","Link"]
    current_header = ws.get("A1:F1")
    if not current_header or current_header[0] != header:
        ws.update("A1:F1", [header])
        ws.format("A1:F1", {'textFormat': {'bold': True}})
    print("[Stage: Sheets] Worksheet ready")
    return ws

def append_to_worksheet(ws, row_dict: dict):
    row_values = [row_dict["ประเภท"], row_dict["ราคา"], row_dict["ทำเล"], row_dict["ขนาด"], row_dict["Owner Score"], row_dict["Link"]]
    ws.append_row(row_values, value_input_option="USER_ENTERED")
    print(f"sheet_appended: {row_dict['Link']}")
    time.sleep(1.2)

def process_and_save(input_path: str, output_path: str, threshold: int = 80, push_to_sheet: bool = True):
    print("[Stage: Input] Load CSV:", input_path)
    df = pd.read_csv(input_path)
    
    if "Post_URL" in df.columns: df.rename(columns={"Post_URL": "URL"}, inplace=True)
    if "Full_Post_Content" in df.columns: df.rename(columns={"Full_Post_Content": "Description"}, inplace=True)
    for col in ["Title", "Property_Details", "Price", "URL", "Description"]:
        if col not in df.columns: df[col] = ""
        df[col] = df[col].fillna('')

    ws = get_worksheet() if push_to_sheet else None
    
    final_rows = []
    print("[Stage: Iterate] Start from index:", RESUME_INDEX)
    
    for i, row in df.iterrows():
        if i < RESUME_INDEX:
            if i % 200 == 0: print(f"resume_skip_index: {i}")
            continue

        full_text = normalize_text(f"{row.get('Title', '')} {row.get('Property_Details', '')} {row.get('Description', '')}")
        
        if not is_relevant_for_sale_post(full_text):
            if i % 50 == 0: print(f"skip_non_relevant_index: {i}")
            continue
        
        user_message = build_user_message(row)
        print(f"gemini_call_index: {i}, URL: {row.get('URL', 'N/A')}")
        llm_result = call_gemini_analyzer(user_message)
        
        if not llm_result:
            print(f"  - Skip: LLM returned empty result for index {i}")
            continue
            
        is_owner = llm_result.get("is_owner")
        confidence = llm_result.get("confidence", 0.0) if llm_result.get("confidence") is not None else 0.0
        score = int(confidence * 100) if is_owner else int((1 - confidence) * 50)
        
        if score >= threshold:
            output_row = build_row(row, llm_result)
            final_rows.append(output_row)
            print(f"  - Accepted: index={i}, score={score}, price={output_row['ราคา']}")
            if push_to_sheet and ws:
                append_to_worksheet(ws, output_row)
        else:
            if i % 20 == 0: print(f"  - Skip: Score below threshold for index {i}, score={score}")

    print("kept_rows:", len(final_rows))
    out_df = pd.DataFrame(final_rows)
    out_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("saved_csv:", output_path)

if __name__ == "__main__":
    input_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\livinginsider_scraped_details.csv"
    output_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\CSV_file\owner_score_gemini_filtered.csv"
    print(f"[Stage: Run] Start pipeline; RESUME_INDEX = {RESUME_INDEX}")
    process_and_save(input_path, output_path, threshold=1, push_to_sheet=True)
    print("[Stage: Done] Completed")

[Stage: Init] Load environment variables
[Stage: Init] Configure Gemini client
[Stage: Run] Start pipeline; RESUME_INDEX = 1514
[Stage: Input] Load CSV: C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\livinginsider_scraped_details.csv
[Stage: Sheets] Authorize and open worksheet
[Stage: Sheets] Worksheet ready
[Stage: Iterate] Start from index: 1514
resume_skip_index: 0
resume_skip_index: 200
resume_skip_index: 400
resume_skip_index: 600
resume_skip_index: 800
resume_skip_index: 1000
resume_skip_index: 1200
resume_skip_index: 1400
gemini_call_index: 1514, URL: https://www.livinginsider.com/livingdetail/2871134/Land-next-to-a-road-80-sq-m-near-Thaweechon-Park-near-Central-Festival-11-km-entering-the-all.html
  - Accepted: index=1514, score=9, price=ขายถูก🚩950,000 | ลดราคาจาก 1.1 ล้าน เหลือ 950,000 ถูกสุดในย่านนี้ | ลดราคาจาก 1.1 ล้าน เหลือ 950,000 ถูกสุดในย่านนี้
sheet_appended: https://www.livinginsider.com/livingdetail/2871134/Land-next-to-a-road-80-sq-m

In [17]:
import csv
import time
import re
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

INPUT_CSV_FILE = Path(r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_listing_urls.csv")
OUTPUT_CSV_FILE = "kaidee_scraped_details.csv"
WAIT = 50

def scrape(driver, url):
    print(f"[Stage: Scrape] Open -> {url}")
    w = WebDriverWait(driver, WAIT)
    driver.get(url)

    btns = driver.find_elements(By.CSS_SELECTOR, "button[aria-label*='cookie i understand'], button[aria-label*='accept'], button:has(span[lang])")
    if btns:
        print("[Stage: Scrape] Click cookie/consent")
        driver.execute_script("arguments[0].click();", btns[0])
        time.sleep(0.3)

    print("[Stage: Scrape] Wait title")
    title_el = w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.sc-747m9u-7")))
    title_txt = title_el.text.strip()

    print("[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present")
    read_more = driver.find_elements(By.XPATH, "//a[contains(normalize-space(),'อ่านเพิ่มเติม')]")
    if read_more:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", read_more[0])
        driver.execute_script("arguments[0].click();", read_more[0])
        time.sleep(0.6)

    print("[Stage: Scrape] Wait price/attributes block")
    w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")))

    price_txt = ""
    price_candidates = driver.find_elements(By.CSS_SELECTOR, "span.sc-3tpgds-0.krrrAv")
    for el in price_candidates:
        t = el.text.strip()
        if re.search(r"\d", t):
            price_txt = t
            break
    if not price_txt:
        block = None
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")
        if blocks:
            block = blocks[0].text
        if block:
            m = re.search(r"([0-9][0-9,\.]{0,18})", block)
            if m:
                price_txt = m.group(1)
    if not price_txt:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[itemprop='price'], meta[property='product:price:amount']")
        if metas:
            v = metas[0].get_attribute("content") or ""
            v = v.strip()
            if v:
                price_txt = v

    print(f"[Stage: Scrape] Price parsed -> '{price_txt}'")

    print("[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)")
    attrs = []
    land_area_els = driver.find_elements(By.XPATH, "//ul[@id='has-attributes']//li[.//span[contains(normalize-space(),'เนื้อที่')]]//span//b")
    if land_area_els:
        v = land_area_els[0].text.strip()
        if v:
            attrs.append(f"เนื้อที่: {v}")
    li = driver.find_elements(By.CSS_SELECTOR, "ul#has-attributes li")
    for x in li:
        t = " ".join(x.text.split())
        if t:
            attrs.append(t)
    attrs_txt = " | ".join(dict.fromkeys([a for a in attrs if a]))

    print("[Stage: Scrape] Collect description (รายละเอียดสินค้า)")
    desc_root = driver.find_elements(By.CSS_SELECTOR, "div.sc-1kndlp1-0")
    if desc_root:
        paras = desc_root[0].find_elements(By.CSS_SELECTOR, "p.inner-text")
        desc_txt = "\n".join(p.text.strip() for p in paras if p.text.strip())
        masked = desc_root[0].find_elements(By.CSS_SELECTOR, "span.masked[data-value]")
        for m in masked:
            mv = (m.get_attribute("data-value") or "").strip()
            mt = (m.text or "").strip()
            if mv and mt:
                desc_txt = desc_txt.replace(mt, mv)
    else:
        desc_txt = ""

    print("[Stage: Scrape] Build Full_Post_Content")
    parts = []
    if price_txt:
        parts.append(f"ราคา: {price_txt}")
    if attrs_txt:
        parts.append(attrs_txt)
    if title_txt:
        parts.append(title_txt)
    if desc_txt:
        parts.append(desc_txt)
    full_text = "\n".join(parts).replace("อ่านเพิ่มเติม", "").replace("ดูน้อยลง", "").strip()
    print(f"[Stage: Scrape] Done -> {len(full_text)} chars")
    return {"Post_URL": url, "Full_Post_Content": full_text}

def main():
    print("[Stage: Init] Validate input CSV path")
    if not INPUT_CSV_FILE.exists():
        print(f"[Stage: Abort] Not found: {INPUT_CSV_FILE}")
        return

    print("[Stage: Init] Launch Chrome")
    options = uc.ChromeOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.page_load_strategy = "eager"
    driver = uc.Chrome(options=options)
    driver.command_executor._client_config.timeout = 180
    driver.set_page_load_timeout(120)
    driver.set_script_timeout(120)

    print("[Stage: Load] Read URLs")
    with open(INPUT_CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [r[0].strip() for r in reader if r and r[0].strip()]
    print(f"[Stage: Load] Total URLs: {len(urls)}")

    print(f"[Stage: Save] Open output CSV for streaming append -> {OUTPUT_CSV_FILE}")
    header = ["Post_URL", "Full_Post_Content"]
    need_header = not Path(OUTPUT_CSV_FILE).exists() or Path(OUTPUT_CSV_FILE).stat().st_size == 0
    with open(OUTPUT_CSV_FILE, "a", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(f_out, fieldnames=header)
        if need_header:
            w.writeheader()
            f_out.flush()
        for i, u in enumerate(urls, start=1):
            print(f"[Stage: Progress] {i}/{len(urls)}")
            row = scrape(driver, u)
            w.writerow(row)
            f_out.flush()
            time.sleep(0.8)

    print("[Stage: Teardown] Quit Chrome")
    driver.quit()
    print("[Stage: Done] kaidee full post content complete and CSV updated per URL")

if __name__ == "__main__":
    main()

[Stage: Init] Validate input CSV path
[Stage: Init] Launch Chrome
[Stage: Load] Read URLs
[Stage: Load] Total URLs: 784
[Stage: Save] Open output CSV for streaming append -> kaidee_scraped_details.csv
[Stage: Progress] 1/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-367612536
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '36,000'
[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)
[Stage: Scrape] Collect description (รายละเอียดสินค้า)
[Stage: Scrape] Build Full_Post_Content
[Stage: Scrape] Done -> 340 chars
[Stage: Progress] 2/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-367612538
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '850,000'
[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)
[Stage: Scr

TimeoutException: Message: timeout: Timed out receiving message from renderer: 42.623
  (Session info: chrome=142.0.7444.163)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xfe4103
	0xfe4144
	0xdee71d
	0xddec5a
	0xdde98d
	0xddc7fe
	0xddd3c7
	0xdea16e
	0xdfc095
	0xe01be6
	0xddda46
	0xdfbe27
	0xe7f14f
	0xe5c706
	0xe2da30
	0xe2ed54
	0x12557b4
	0x125098a
	0x100c392
	0xffc4c8
	0x100324d
	0xfec478
	0xfec63c
	0xfd67ca
	0x750e5d49
	0x76fed6db
	0x76fed661


In [18]:
import csv
import time
import re
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

INPUT_CSV_FILE = Path(r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_listing_urls.csv")
OUTPUT_CSV_FILE = "kaidee_scraped_details.csv"
WAIT = 50
START_INDEX = 575

def scrape(driver, url):
    print(f"[Stage: Scrape] Open -> {url}")
    w = WebDriverWait(driver, WAIT)
    driver.get(url)

    btns = driver.find_elements(By.CSS_SELECTOR, "button[aria-label*='cookie i understand'], button[aria-label*='accept'], button:has(span[lang])")
    if btns:
        print("[Stage: Scrape] Click cookie/consent")
        driver.execute_script("arguments[0].click();", btns[0])
        time.sleep(0.3)

    print("[Stage: Scrape] Wait title")
    title_el = w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.sc-747m9u-7")))
    title_txt = title_el.text.strip()

    print("[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present")
    read_more = driver.find_elements(By.XPATH, "//a[contains(normalize-space(),'อ่านเพิ่มเติม')]")
    if read_more:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", read_more[0])
        driver.execute_script("arguments[0].click();", read_more[0])
        time.sleep(0.6)

    print("[Stage: Scrape] Wait price/attributes block")
    w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")))

    price_txt = ""
    price_candidates = driver.find_elements(By.CSS_SELECTOR, "span.sc-3tpgds-0.krrrAv")
    for el in price_candidates:
        t = el.text.strip()
        if re.search(r"\d", t):
            price_txt = t
            break
    if not price_txt:
        block = None
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")
        if blocks:
            block = blocks[0].text
        if block:
            m = re.search(r"([0-9][0-9,\.]{0,18})", block)
            if m:
                price_txt = m.group(1)
    if not price_txt:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[itemprop='price'], meta[property='product:price:amount']")
        if metas:
            v = metas[0].get_attribute("content") or ""
            v = v.strip()
            if v:
                price_txt = v

    print(f"[Stage: Scrape] Price parsed -> '{price_txt}'")

    print("[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)")
    attrs = []
    land_area_els = driver.find_elements(By.XPATH, "//ul[@id='has-attributes']//li[.//span[contains(normalize-space(),'เนื้อที่')]]//span//b")
    if land_area_els:
        v = land_area_els[0].text.strip()
        if v:
            attrs.append(f"เนื้อที่: {v}")
    li = driver.find_elements(By.CSS_SELECTOR, "ul#has-attributes li")
    for x in li:
        t = " ".join(x.text.split())
        if t:
            attrs.append(t)
    attrs_txt = " | ".join(dict.fromkeys([a for a in attrs if a]))

    print("[Stage: Scrape] Collect description (รายละเอียดสินค้า)")
    desc_root = driver.find_elements(By.CSS_SELECTOR, "div.sc-1kndlp1-0")
    if desc_root:
        paras = desc_root[0].find_elements(By.CSS_SELECTOR, "p.inner-text")
        desc_txt = "\n".join(p.text.strip() for p in paras if p.text.strip())
        masked = desc_root[0].find_elements(By.CSS_SELECTOR, "span.masked[data-value]")
        for m in masked:
            mv = (m.get_attribute("data-value") or "").strip()
            mt = (m.text or "").strip()
            if mv and mt:
                desc_txt = desc_txt.replace(mt, mv)
    else:
        desc_txt = ""

    print("[Stage: Scrape] Build Full_Post_Content")
    parts = []
    if price_txt:
        parts.append(f"ราคา: {price_txt}")
    if attrs_txt:
        parts.append(attrs_txt)
    if title_txt:
        parts.append(title_txt)
    if desc_txt:
        parts.append(desc_txt)
    full_text = "\n".join(parts).replace("อ่านเพิ่มเติม", "").replace("ดูน้อยลง", "").strip()
    print(f"[Stage: Scrape] Done -> {len(full_text)} chars")
    return {"Post_URL": url, "Full_Post_Content": full_text}

def main():
    print("[Stage: Init] Validate input CSV path")
    if not INPUT_CSV_FILE.exists():
        print(f"[Stage: Abort] Not found: {INPUT_CSV_FILE}")
        return

    print("[Stage: Init] Launch Chrome")
    options = uc.ChromeOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.page_load_strategy = "eager"
    driver = uc.Chrome(options=options)
    driver.command_executor._client_config.timeout = 180
    driver.set_page_load_timeout(120)
    driver.set_script_timeout(120)

    print("[Stage: Load] Read URLs")
    with open(INPUT_CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [r[0].strip() for r in reader if r and r[0].strip()]
    print(f"[Stage: Load] Total URLs: {len(urls)}")

    urls_to_process = urls[START_INDEX-1:]
    print(f"[Stage: Resume] Start at URL index {START_INDEX}, remaining {len(urls_to_process)}")

    print(f"[Stage: Save] Open output CSV for streaming append -> {OUTPUT_CSV_FILE}")
    header = ["Post_URL", "Full_Post_Content"]
    need_header = not Path(OUTPUT_CSV_FILE).exists() or Path(OUTPUT_CSV_FILE).stat().st_size == 0
    with open(OUTPUT_CSV_FILE, "a", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(f_out, fieldnames=header)
        if need_header:
            w.writeheader()
            f_out.flush()
        for i, u in enumerate(urls_to_process, start=START_INDEX):
            print(f"[Stage: Progress] {i}/{len(urls)}")
            row = scrape(driver, u)
            w.writerow(row)
            f_out.flush()
            time.sleep(0.8)

    print("[Stage: Teardown] Quit Chrome")
    driver.quit()
    print("[Stage: Done] kaidee full post content complete and CSV updated per URL")

if __name__ == "__main__":
    main()


[Stage: Init] Validate input CSV path
[Stage: Init] Launch Chrome
[Stage: Load] Read URLs
[Stage: Load] Total URLs: 784
[Stage: Resume] Start at URL index 575, remaining 210
[Stage: Save] Open output CSV for streaming append -> kaidee_scraped_details.csv
[Stage: Progress] 575/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-371048608
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '33,000'
[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)
[Stage: Scrape] Collect description (รายละเอียดสินค้า)
[Stage: Scrape] Build Full_Post_Content
[Stage: Scrape] Done -> 1799 chars
[Stage: Progress] 576/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-371048740
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '20,000'
[Stage: Scrape] Collect at